# The Feature That Splits Everything Perfectly

**Topic:** Impurity and information gain &nbsp;·&nbsp; **Difficulty:** Medium

One of your columns is a customer ID. Information gain adores it: split the node
one branch per distinct value and every branch holds a single row, so the
impurity after the split is exactly zero and the gain is the largest possible.
The tree picks it first, every time, and the model is worthless on any customer
it has not already met.

This is the classic bias of multi-way information gain toward high-cardinality
attributes. Implement the split search, watch the ID win, then try the two
standard defences: one narrows its lead and still loses to it, and the other is
decisive.

## The data (created in the next cell)

- `X` `(800, 5)`: all columns are **categorical** integer codes.
- Column 0 is `customer_id` with 800 distinct values (one per row).
- Columns 1-4 have 2 to 6 levels each, and only columns 1 and 2 relate to the label.
- `y` `(800,)`: binary.

## Your task

With **NumPy only**, using a **multi-way** split (one branch per level):

1. Implement entropy in bits and the weighted entropy after splitting on a
   feature. Information gain is `H(parent) - weighted H(children)`.
2. `gains`: the information gain of each of the 5 features.
3. `gain_ratios`: gain divided by the split's own entropy (the entropy of the
   branch-size distribution), the C4.5 correction. Guard against dividing by
   zero. Note what this does and does not fix.
4. `best_min_leaf`: the column index (0 to 4) of the feature with the highest
   gain among those whose every branch holds at least **20** rows.
5. Set `answers` to a dict with `"gains"` `(5,)`, `"gain_ratios"` `(5,)`,
   `"best_by_gain"` (int), `"best_by_ratio"` (int), `"best_min_leaf"` (int).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks raw gain picks the ID, that gain ratio cuts its lead by more than 10x
without dethroning it, and that the minimum branch size finally rejects it.

In [ ]:
import numpy as np

rng = np.random.default_rng(1414)
n = 800
customer_id = np.arange(n)
f1 = rng.integers(0, 4, n)
f2 = rng.integers(0, 3, n)
f3 = rng.integers(0, 6, n)
f4 = rng.integers(0, 2, n)
logit = 1.3 * (f1 - 1.5) - 1.6 * (f2 - 1.0)
y = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
X = np.column_stack([customer_id, f1, f2, f3, f4])
print(f"X {X.shape}, distinct values per column: "
      f"{[len(np.unique(X[:, j])) for j in range(X.shape[1])]}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("gains", [5], 1.278032744204364, -0.2899742458532456, 1e-06),
    ("gain_ratios", [5], 0.25780172516332256, -0.10045950445238871, 1e-06),
    ("best_by_gain", [], 0.0, 0.0, 1e-06),
    ("best_by_ratio", [], 0.0, 0.0, 1e-06),
    ("best_min_leaf", [], 1.0, -0.16743801481449916, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "best_by_gain" in answers:
    _g = np.asarray(answers["gains"], dtype=float)
    _r = np.asarray(answers["gain_ratios"], dtype=float)
    _lead_gain = _g[0] - np.max(_g[1:])
    _lead_ratio = _r[0] - np.max(_r[1:])
    _id_wins = int(answers["best_by_gain"]) == 0
    _narrowed = _lead_ratio * 10 < _lead_gain
    _leaf_ok = int(answers["best_min_leaf"]) != 0
    print(f"  {'raw gain picks the ID':<24} {'ok' if _id_wins else 'wrong'}")
    print(f"  {'gain ratio narrows lead':<24} {'ok' if _narrowed else 'wrong'}  "
          f"({_lead_gain:.3f} -> {_lead_ratio:.3f}, still first)")
    print(f"  {'min branch rejects it':<24} {'ok' if _leaf_ok else 'wrong'}")
    _ok = _ok and _id_wins and _narrowed and _leaf_ok

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Splitting on the ID gives 800 branches of one row each. A single-row branch is
pure, so the weighted child entropy is 0 and the gain equals the parent entropy,
the maximum any split can achieve. It generalizes to nothing. Gain ratio divides by the split's own entropy, which for 800 equal branches is
`log2(800) = 9.64` bits. That cuts the ID from 0.997 to 0.103 and shrinks its
lead over the best real feature from 0.824 to 0.017, a big improvement that
still ranks the identifier first. Against a *perfect* identifier the correction
is not enough, and only the minimum branch size removes it outright. In practice
you drop identifier columns before training rather than relying on either.

```python
import numpy as np

def entropy(labels):
    if len(labels) == 0:
        return 0.0
    p = labels.mean()
    out = 0.0
    for q in (p, 1 - p):
        if q > 0:
            out -= q * np.log2(q)
    return float(out)

parent = entropy(y)
n = len(y)

gains, ratios, min_branch = [], [], []
for j in range(X.shape[1]):
    levels = np.unique(X[:, j])
    child, split_info, smallest = 0.0, 0.0, n
    for v in levels:
        m = X[:, j] == v
        w = m.sum() / n
        child += w * entropy(y[m])
        if w > 0:
            split_info -= w * np.log2(w)
        smallest = min(smallest, int(m.sum()))
    g = parent - child
    gains.append(g)
    ratios.append(g / split_info if split_info > 1e-12 else 0.0)
    min_branch.append(smallest)

gains = np.array(gains)
ratios = np.array(ratios)
min_branch = np.array(min_branch)

eligible = min_branch >= 20
masked = np.where(eligible, gains, -np.inf)

answers = {
    "gains": gains,
    "gain_ratios": ratios,
    "best_by_gain": int(np.argmax(gains)),
    "best_by_ratio": int(np.argmax(ratios)),
    "best_min_leaf": int(np.argmax(masked)),
}
```